# ASL MNIST — analiza wyników

Ten notebook wczytuje wyniki zapisane przez `02_model_training.ipynb` i porównuje konfiguracje MLP, LeNet oraz CNN.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

from configs.constants import EXPERIMENT_HISTORY_CSV, EXPERIMENT_RESULTS_CSV
from src.helpers import plot_experiment_comparison

In [ ]:
results_path = PROJECT_ROOT / EXPERIMENT_RESULTS_CSV
history_path = PROJECT_ROOT / EXPERIMENT_HISTORY_CSV

if not results_path.exists() or not history_path.exists():
    raise FileNotFoundError(
        "Brakuje plików z wynikami. Najpierw uruchom notebook "
        "notebooks/02_model_training.ipynb."
    )

results_df = pd.read_csv(results_path)
history_df = pd.read_csv(history_path)

results_df.sort_values("best_val_acc", ascending=False)

## Najlepsza konfiguracja

Ranking bazuje na najwyższej walidacyjnej accuracy uzyskanej w trakcie treningu.

In [ ]:
display(plot_experiment_comparison(
    results_df.sort_values("best_val_acc", ascending=False),
    metric="best_val_acc",
    title="Najlepsza walidacyjna accuracy",
))

## Wyniki według typu modelu

Ten widok pozwala szybko porównać, czy w obecnym sweepie lepiej wypadają konfiguracje MLP, LeNet czy CNN.

In [ ]:
summary_by_model = (
    results_df.groupby("model")
    .agg(
        best_val_acc=("best_val_acc", "max"),
        mean_best_val_acc=("best_val_acc", "mean"),
        experiments=("name", "count"),
    )
    .reset_index()
    .sort_values("best_val_acc", ascending=False)
)

summary_by_model

## Najlepszy wynik per model

Wybieramy najlepszą konfigurację osobno dla MLP, LeNet i CNN. To daje szybki obraz, który typ architektury ma największy potencjał przy obecnym sweepie.

In [ ]:
required_columns = [
    "model",
    "name",
    "optimizer",
    "learning_rate",
    "momentum",
    "best_val_acc",
    "best_epoch",
]
optional_columns = ["total_params"] if "total_params" in results_df.columns else []

best_per_model = (
    results_df.sort_values("best_val_acc", ascending=False)
    .groupby("model", as_index=False)
    .first()
    [required_columns + optional_columns]
    .sort_values("best_val_acc", ascending=False)
)

best_per_model

## Porównanie tych samych ustawień

Porównujemy modele przy tych samych ustawieniach optymalizatora. Dzięki temu widać, czy różnica wynika głównie z architektury, czy z parametrów treningu.

In [ ]:
comparison_df = results_df.copy()
comparison_df["training_setup"] = comparison_df.apply(
    lambda row: f"{row['optimizer']} | lr={row['learning_rate']} | momentum={row['momentum']}",
    axis=1,
)

model_comparison = comparison_df.pivot_table(
    index="training_setup",
    columns="model",
    values="best_val_acc",
    aggfunc="max",
).reset_index()

model_comparison

## Accuracy względem liczby parametrów

Ten wykres pokazuje kompromis między jakością a rozmiarem modelu. LeNet powinien być interesujący, jeśli daje wynik bliższy CNN przy znacznie mniejszej liczbie parametrów.

In [ ]:
if "total_params" not in results_df.columns:
    print("Brakuje kolumny total_params. Uruchom ponownie 02_model_training.ipynb.")
else:
    fig = go.Figure()
    for model_name, group in results_df.groupby("model"):
        fig.add_trace(
            go.Scatter(
                x=group["total_params"],
                y=group["best_val_acc"],
                mode="markers+text",
                name=model_name,
                text=group["name"],
                textposition="top center",
                marker=dict(size=11),
                hovertemplate=(
                    "%{text}<br>"
                    "Params: %{x}<br>"
                    "Best val acc: %{y:.4f}<extra></extra>"
                ),
            )
        )

    fig.update_layout(
        template="plotly_white",
        title="Best val accuracy vs liczba parametrów",
        xaxis_title="Liczba parametrów",
        yaxis_title="Best val accuracy",
        yaxis_range=[0, 1],
        font=dict(family="Segoe UI, Arial, Helvetica, sans-serif", size=14),
        legend=dict(orientation="h", y=1.08),
    )
    display(fig)

## Accuracy w epokach

Ten wykres pokazuje, jak szybko poszczególne konfiguracje dochodziły do dobrych wyników i czy wynik był stabilny między epokami.

In [ ]:
fig = go.Figure()
for name, group in history_df.groupby("name"):
    model_name = group["model"].iloc[0]
    fig.add_trace(
        go.Scatter(
            x=group["epoch"],
            y=group["val_acc"],
            mode="lines+markers",
            name=f"{name} ({model_name})",
        )
    )

fig.update_layout(
    template="plotly_white",
    title="Val accuracy per epoka",
    xaxis_title="Epoka",
    yaxis_title="Val accuracy",
    yaxis_range=[0, 1],
    font=dict(family="Segoe UI, Arial, Helvetica, sans-serif", size=14),
    legend=dict(orientation="h", y=1.08),
)
display(fig)

## Loss w epokach

Loss pomaga zobaczyć, czy konfiguracja nadal się uczyła, czy zaczęła zachowywać się niestabilnie.

In [ ]:
fig = go.Figure()
for name, group in history_df.groupby("name"):
    model_name = group["model"].iloc[0]
    fig.add_trace(
        go.Scatter(
            x=group["epoch"],
            y=group["val_loss"],
            mode="lines+markers",
            name=f"{name} ({model_name})",
        )
    )

fig.update_layout(
    template="plotly_white",
    title="Val loss per epoka",
    xaxis_title="Epoka",
    yaxis_title="Val loss",
    font=dict(family="Segoe UI, Arial, Helvetica, sans-serif", size=14),
    legend=dict(orientation="h", y=1.08),
)
display(fig)

## Podsumowanie

Przy 4 epokach traktuj wyniki jako szybki screening, nie finalne porównanie. W kolejnym kroku warto zwiększyć liczbę epok dla 1-2 najlepszych konfiguracji.

In [ ]:
best = results_df.sort_values("best_val_acc", ascending=False).iloc[0]
print(f"Najlepszy eksperyment: {best['name']}")
print(f"Model: {best['model']}")
print(f"Optimizer: {best['optimizer']}, lr={best['learning_rate']}, momentum={best['momentum']}")
print(f"Best val acc: {best['best_val_acc']:.4f} w epoce {int(best['best_epoch'])}")
print(f"Checkpoint: {best['checkpoint_path']}")